# CoT and DFA State Tracking in a One-Block Transformer


In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, pairwise_distances

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
TASK_NAMES = ["parity", "contains_11", "mod3"]

TRAIN_MIN_LEN = 2
TRAIN_MAX_LEN = 32
OOD_MIN_LEN = 33
OOD_MAX_LEN = 64

TRAIN_SAMPLES = 20000
IID_TEST_SAMPLES = 3000
OOD_TEST_SAMPLES = 3000

PER_LENGTH_MIN = 2
PER_LENGTH_MAX = 128
PER_LENGTH_SAMPLES = 300

MAX_ATTENTION_STRINGS = 250
MAX_GEOMETRY_STRINGS = 500

N_EPOCHS = 20
BATCH_SIZE = 64
LR = 3e-4
D_MODEL = 128
N_HEADS = 2
DROPOUT = 0.1

MODEL_KINDS = ["direct", "cot_true", "cot_pad"]

RESULTS_CSV = "analysis/results.csv"
PER_LENGTH_CSV = "analysis/per_length_results.csv"
ATTENTION_CSV = "analysis/attention_results.csv"
ABLATION_CSV = "analysis/ablation_results.csv"
GEOMETRY_CSV = "analysis/geometry_results.csv"

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# PARITY: state 0 = even number of ones so far, state 1 = odd.
def parity_label(s):
    return int(sum(s) % 2 == 0)

def parity_transition(tok, state):
    return state ^ tok

def parity_state_to_label(state):
    return 1 if state == 0 else 0


# CONTAINS-11: state 0 = no 1 yet, state 1 = previous token was 1, state 2 = seen 11.
def contains_11_label(s):
    return int(any(s[i] == 1 and s[i + 1] == 1 for i in range(len(s) - 1)))

def contains_11_transition(tok, state):
    if state == 2:
        return 2
    if state == 0:
        return 1 if tok == 1 else 0
    if state == 1:
        return 2 if tok == 1 else 0

def contains_11_state_to_label(state):
    return 1 if state == 2 else 0


# MOD3: state = number of ones seen so far, modulo 3.
def mod3_label(s):
    return sum(s) % 3

def mod3_transition(tok, state):
    return (state + tok) % 3

def mod3_state_to_label(state):
    return state


TASKS = {
    "parity": {
        "label_fn": parity_label,
        "transition_fn": parity_transition,
        "s2l": parity_state_to_label,
        "n_states": 2,
        "n_classes": 2,
    },
    "contains_11": {
        "label_fn": contains_11_label,
        "transition_fn": contains_11_transition,
        "s2l": contains_11_state_to_label,
        "n_states": 3,
        "n_classes": 2,
    },
    "mod3": {
        "label_fn": mod3_label,
        "transition_fn": mod3_transition,
        "s2l": mod3_state_to_label,
        "n_states": 3,
        "n_classes": 3,
    },
}


def run_dfa(string, transition_fn, start=0):
    states = []
    state = start
    for tok in string:
        state = transition_fn(tok, state)
        states.append(state)
    return states

In [ ]:
def make_no_11_string(length, rng, p_one=0.5):
    s = []
    for _ in range(length):
        if s and s[-1] == 1:
            s.append(0)
        else:
            s.append(1 if rng.random() < p_one else 0)
    return s


def sample_string(length, rng, p_one=0.5):
    return [1 if rng.random() < p_one else 0 for _ in range(length)]


def generate_dataset(min_len, max_len, n, label_fn, task_name, n_classes, seed, p_one=0.5, balance_labels=True):
    rng = random.Random(seed)
    out = []
    counts = {c: 0 for c in range(n_classes)}
    target_per_class = math.ceil(n / n_classes)

    attempts = 0
    max_attempts = n * 500
    while len(out) < n and attempts < max_attempts:
        attempts += 1
        length = rng.randint(min_len, max_len)

        if task_name == "contains_11" and balance_labels:
            if counts[0] < target_per_class and (counts[1] >= target_per_class or rng.random() < 0.5):
                s = make_no_11_string(length, rng, p_one=min(p_one, 0.75))
            else:
                s = sample_string(length, rng, p_one)
        else:
            s = sample_string(length, rng, p_one)

        label = label_fn(s)
        if not balance_labels or counts.get(label, 0) < target_per_class:
            out.append({"string": s, "class": label, "length": len(s)})
            counts[label] = counts.get(label, 0) + 1

    rng.shuffle(out)
    return out[:n]


def split_xy(data):
    return [d["string"] for d in data], [d["class"] for d in data]

In [ ]:
BASE_VOCAB = {0: 0, 1: 1, "[PAD]": 2, "[CLS]": 3}

def make_cot_vocab(n_states):
    vocab = dict(BASE_VOCAB)
    for st in range(n_states):
        vocab[f"s{st}"] = len(vocab)
    return vocab


def tokenise(tokens, vocab, max_length):
    seq = ["[CLS]"] + list(tokens)
    ids = [vocab[t] for t in seq]
    mask = [1] * len(ids)
    pad = max_length - len(ids)
    ids += [vocab["[PAD]"]] * pad
    mask += [0] * pad
    return {"input_ids": ids, "attention_mask": mask}


def build_interleaved_sequence(string, transition_fn, scratchpad_mode, n_states, rng):
    state = 0
    interleaved = []
    labels = [-1]  # CLS position

    for tok in string:
        interleaved.append(tok)
        state = transition_fn(tok, state)
        labels.append(state)  # supervise at the real token position

        if scratchpad_mode == "true":
            scratch_state = state
        elif scratchpad_mode == "pad":
            scratch_state = 0
        else:
            scratch_state = rng.randint(0, n_states - 1)

        interleaved.append(f"s{scratch_state}")
        labels.append(-1)  # never supervise the scratchpad token itself

    return interleaved, labels


def real_token_positions(labels):
    return [i for i, y in enumerate(labels) if y != -1]


def max_base_len(max_string_len):
    return max_string_len + 1


def max_cot_len(max_string_len):
    return 2 * max_string_len + 1


class BaselineDataset(Dataset):
    def __init__(self, strings, labels, vocab, max_len):
        self.samples = []
        for s, y in zip(strings, labels):
            tok = tokenise(s, vocab, max_len)
            self.samples.append({
                "input_ids": torch.tensor(tok["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(tok["attention_mask"], dtype=torch.long),
                "label": torch.tensor(y, dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class CoTDataset(Dataset):
    def __init__(self, strings, transition_fn, vocab, max_len, scratchpad_mode, n_states, seed=0):
        self.samples = []
        rng = random.Random(seed)
        for s in strings:
            inter, labels = build_interleaved_sequence(s, transition_fn, scratchpad_mode, n_states, rng)
            tok = tokenise(inter, vocab, max_len)
            labels = labels + [-1] * (max_len - len(labels))
            self.samples.append({
                "input_ids": torch.tensor(tok["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(tok["attention_mask"], dtype=torch.long),
                "state_labels": torch.tensor(labels, dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def make_direct_loader(strings, labels, max_len, batch_size, shuffle=True):
    ds = BaselineDataset(strings, labels, BASE_VOCAB, max_len)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def make_cot_loader(strings, transition_fn, vocab, max_len, scratchpad_mode, n_states, batch_size, seed, shuffle=True):
    ds = CoTDataset(strings, transition_fn, vocab, max_len, scratchpad_mode, n_states, seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class OneBlockTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_classes, max_len=512, dropout=0.1, pad_id=2, state_token_ids=None):
        super().__init__()
        self.pad_id = pad_id
        self.state_token_ids = set(state_token_ids or [])

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len)

        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

        self.mlp_fc1 = nn.Linear(d_model, d_model * 4)
        self.mlp_gelu = nn.GELU()
        self.mlp_drop1 = nn.Dropout(dropout)
        self.mlp_fc2 = nn.Linear(d_model * 4, d_model)
        self.mlp_drop2 = nn.Dropout(dropout)

        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, input_ids, attention_mask, causal=False, return_all=False,
                ablate_attn=False, ablate_mlp=False, ablate_state_embeddings=False, ablate_pos=False):
        emb = self.embedding(input_ids)

        if ablate_state_embeddings and self.state_token_ids:
            state_mask = torch.zeros_like(input_ids, dtype=torch.bool)
            for sid in self.state_token_ids:
                state_mask |= (input_ids == sid)
            emb = emb.masked_fill(state_mask.unsqueeze(-1), 0.0)

        x = emb if ablate_pos else self.pos_enc(emb)
        key_padding_mask = (attention_mask == 0)

        attn_mask = None
        if causal:
            L = input_ids.size(1)
            attn_mask = torch.triu(torch.ones(L, L, device=input_ids.device), diagonal=1).bool()

        attn_out, attn_weights = self.attention(
            x, x, x, key_padding_mask=key_padding_mask, attn_mask=attn_mask,
            need_weights=True, average_attn_weights=False,
        )
        if ablate_attn:
            attn_out = torch.zeros_like(attn_out)
        x = self.norm1(x + self.drop(attn_out))

        pre = self.mlp_fc1(x)
        post_gelu = self.mlp_gelu(pre)
        mlp_out = self.mlp_fc2(self.mlp_drop1(post_gelu))
        if ablate_mlp:
            mlp_out = torch.zeros_like(mlp_out)
        x_final = self.norm2(x + self.mlp_drop2(mlp_out))

        if return_all:
            return x_final, {"attn_weights": attn_weights}
        return x_final

    def logits_at_cls(self, input_ids, attention_mask, **kwargs):
        x = self.forward(input_ids, attention_mask, causal=False, **kwargs)
        return self.classifier(x[:, 0, :])

    def logits_all_positions(self, input_ids, attention_mask, causal=True, **kwargs):
        x = self.forward(input_ids, attention_mask, causal=causal, **kwargs)
        return self.classifier(x)

In [ ]:
def train_direct(model, train_loader, n_epochs, lr, device):
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = CrossEntropyLoss()

    for epoch in range(n_epochs):
        model.train()
        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["label"].to(device)

            opt.zero_grad()
            loss = crit(model.logits_at_cls(ids, mask), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

    return model


def train_cot_state(model, train_loader, n_epochs, lr, device, n_states):
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = CrossEntropyLoss(ignore_index=-1)

    for epoch in range(n_epochs):
        model.train()
        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["state_labels"].to(device)

            opt.zero_grad()
            logits = model.logits_all_positions(ids, mask)
            loss = crit(logits.view(-1, n_states), labels.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

    return model


@torch.no_grad()
def eval_direct_accuracy(model, loader, device, **forward_kwargs):
    model.eval()
    ok = total = 0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = batch["label"].to(device)
        pred = model.logits_at_cls(ids, mask, **forward_kwargs).argmax(dim=-1)
        ok += (pred == y).sum().item()
        total += y.numel()
    return ok / max(total, 1)


@torch.no_grad()
def eval_cot_state_accuracy(model, loader, device, **forward_kwargs):
    model.eval()
    ok = total = 0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["state_labels"].to(device)
        valid = labels != -1
        pred = model.logits_all_positions(ids, mask, **forward_kwargs).argmax(dim=-1)
        ok += (pred[valid] == labels[valid]).sum().item()
        total += valid.sum().item()
    return ok / max(total, 1)


@torch.no_grad()
def eval_cot_final_answer_accuracy(model, loader, device, s2l, **forward_kwargs):
    model.eval()
    ok = total = 0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["state_labels"].to(device)
        pred_states = model.logits_all_positions(ids, mask, **forward_kwargs).argmax(dim=-1)

        for i in range(ids.size(0)):
            valid_positions = torch.where(labels[i] != -1)[0]
            if len(valid_positions) == 0:
                continue
            final_pos = valid_positions[-1]
            pred_label = s2l(int(pred_states[i, final_pos].item()))
            true_label = s2l(int(labels[i, final_pos].item()))
            ok += int(pred_label == true_label)
            total += 1
    return ok / max(total, 1)


@torch.no_grad()
def eval_transition_consistency(model, strings, transition_fn, vocab, max_len, device, scratchpad_mode, n_states, seed=0, batch_size=64):
    model.eval()
    ok = total = 0
    rng = random.Random(seed)

    for start in range(0, len(strings), batch_size):
        batch_strings = strings[start:start + batch_size]
        ids_list, mask_list, positions_list = [], [], []

        for string in batch_strings:
            inter, labels = build_interleaved_sequence(string, transition_fn, scratchpad_mode, n_states, rng)
            tok = tokenise(inter, vocab, max_len)
            labels = labels + [-1] * (max_len - len(labels))
            real_positions = real_token_positions(labels)
            ids_list.append(torch.tensor(tok["input_ids"], dtype=torch.long))
            mask_list.append(torch.tensor(tok["attention_mask"], dtype=torch.long))
            positions_list.append(real_positions)

        ids = torch.stack(ids_list, dim=0).to(device)
        mask = torch.stack(mask_list, dim=0).to(device)
        pred_states = model.logits_all_positions(ids, mask).argmax(dim=-1)

        for b_idx, (string, real_positions) in enumerate(zip(batch_strings, positions_list)):
            prev_pred_state = 0
            for step, pos in enumerate(real_positions):
                expected = transition_fn(string[step], prev_pred_state)
                predicted = int(pred_states[b_idx, pos].item())
                ok += int(predicted == expected)
                total += 1
                prev_pred_state = predicted

    return ok / max(total, 1)

In [ ]:
@torch.no_grad()
def collect_geometry_dataset(model, strings, labels, transition_fn, vocab, max_len, mode, device, scratchpad_mode="true", n_states=2, seed=0, batch_size=64):
    # Hidden states at real token positions: direct = bit positions after [CLS],
    # cot = bit positions in the interleaved bit/state sequence.
    model.eval()
    X_hidden = []
    rows = []
    rng = random.Random(seed)

    for start in range(0, len(strings), batch_size):
        batch_strings = strings[start:start + batch_size]
        batch_labels = labels[start:start + batch_size]
        ids_list, mask_list, positions_list, states_list = [], [], [], []

        for string in batch_strings:
            states = run_dfa(string, transition_fn)
            states_list.append(states)

            if mode == "direct":
                tok = tokenise(string, vocab, max_len)
                positions = list(range(1, len(string) + 1))
            else:
                inter, state_labels = build_interleaved_sequence(string, transition_fn, scratchpad_mode, n_states, rng)
                tok = tokenise(inter, vocab, max_len)
                state_labels = state_labels + [-1] * (max_len - len(state_labels))
                positions = real_token_positions(state_labels)

            ids_list.append(torch.tensor(tok["input_ids"], dtype=torch.long))
            mask_list.append(torch.tensor(tok["attention_mask"], dtype=torch.long))
            positions_list.append(positions)

        ids = torch.stack(ids_list, dim=0).to(device)
        mask = torch.stack(mask_list, dim=0).to(device)
        x = model.forward(ids, mask, causal=(mode != "direct"))

        for b_idx, (string, label, positions, states) in enumerate(zip(batch_strings, batch_labels, positions_list, states_list)):
            string_len = len(string)
            n_ones = int(sum(string))
            for step, pos in enumerate(positions):
                if step >= len(states):
                    break
                h = x[b_idx, pos, :].detach().cpu().numpy()
                X_hidden.append(h)
                rows.append({
                    "position": step + 1,
                    "input_length": string_len,
                    "n_ones": n_ones,
                    "true_state": int(states[step]),
                    "label": int(label),
                })

    return np.asarray(X_hidden), pd.DataFrame(rows)


def cluster_purity_score(y_true, y_cluster):
    if len(y_true) == 0:
        return float("nan")
    total = 0
    for cluster_id in np.unique(y_cluster):
        mask = (y_cluster == cluster_id)
        if not np.any(mask):
            continue
        _, counts = np.unique(y_true[mask], return_counts=True)
        total += int(counts.max())
    return float(total / len(y_true))


def within_between_distance_stats(X, y, seed, max_points=2500):
    if len(X) < 3 or len(np.unique(y)) < 2:
        return {"within_state_distance": float("nan"), "between_state_distance": float("nan"), "between_within_ratio": float("nan")}

    rng = np.random.default_rng(seed)
    if len(X) > max_points:
        idx = rng.choice(len(X), size=max_points, replace=False)
        X_s, y_s = X[idx], y[idx]
    else:
        X_s, y_s = X, y

    D = pairwise_distances(X_s, metric="euclidean")
    upper = np.triu_indices_from(D, k=1)
    same = y_s[upper[0]] == y_s[upper[1]]
    diff = ~same

    within = float(np.mean(D[upper][same])) if np.any(same) else float("nan")
    between = float(np.mean(D[upper][diff])) if np.any(diff) else float("nan")
    ratio = float(between / within) if within and not np.isnan(within) else float("nan")

    return {"within_state_distance": within, "between_state_distance": between, "between_within_ratio": ratio}


def run_representational_geometry_analysis(model, strings, labels, transition_fn, vocab, max_len, mode, device,
                                            task_name, model_kind, seed, scratchpad_mode, n_states):
    X, meta = collect_geometry_dataset(model, strings, labels, transition_fn, vocab, max_len, mode, device, scratchpad_mode, n_states, seed)

    if len(X) < max(10, n_states * 5) or len(meta) == 0:
        return [{"task": task_name, "seed": seed, "model": model_kind, "metric": "geometry_skipped_too_few_points", "value": 1.0}]

    y_state = meta["true_state"].to_numpy(dtype=int)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2, random_state=seed)
    pca.fit(X_scaled)
    pca_var = pca.explained_variance_ratio_

    kmeans = KMeans(n_clusters=n_states, random_state=seed, n_init=20)
    clusters = kmeans.fit_predict(X_scaled)

    ari = float(adjusted_rand_score(y_state, clusters))
    nmi = float(normalized_mutual_info_score(y_state, clusters))
    purity = cluster_purity_score(y_state, clusters)

    silhouette = float("nan")
    if len(np.unique(clusters)) > 1 and len(X_scaled) > len(np.unique(clusters)):
        try:
            sample_size = min(2000, len(X_scaled))
            silhouette = float(silhouette_score(X_scaled, clusters, sample_size=sample_size, random_state=seed))
        except Exception:
            silhouette = float("nan")

    dist_stats = within_between_distance_stats(X_scaled, y_state, seed=seed)

    metric_values = {
        "pca_explained_variance_pc1": float(pca_var[0]),
        "pca_explained_variance_pc2": float(pca_var[1]),
        "kmeans_ari_true_state": ari,
        "kmeans_nmi_true_state": nmi,
        "kmeans_purity_true_state": purity,
        "kmeans_silhouette": silhouette,
        **dist_stats,
    }

    rows = []
    for metric, value in metric_values.items():
        rows.append({"task": task_name, "seed": seed, "model": model_kind, "metric": metric, "value": value, "n_points": len(X)})
    return rows

In [ ]:
def _attention_entropy(attn_row, valid_len):
    # normalised entropy: 1.0 = diffuse/uniform, 0.0 = maximally concentrated.
    probs = np.asarray(attn_row[:valid_len], dtype=float)
    probs = probs / max(probs.sum(), 1e-12)
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    norm_entropy = 0.0 if valid_len <= 1 else float(entropy / np.log(valid_len))
    return entropy, norm_entropy, 1.0 - norm_entropy


@torch.no_grad()
def attention_routing_analysis(model, strings, transition_fn, vocab, max_len, device, scratchpad_mode, n_states, seed, batch_size=64):
    # Measures where attention goes from real bit positions in a CoT sequence.
    # For bit position p, p-1 is the previous scratchpad state token, p-2 the previous bit.
    model.eval()
    rng = random.Random(seed)
    accum = {}
    count = 0

    for start in range(0, len(strings), batch_size):
        batch_strings = strings[start:start + batch_size]
        ids_list, mask_list, positions_list = [], [], []

        for string in batch_strings:
            inter, labels = build_interleaved_sequence(string, transition_fn, scratchpad_mode, n_states, rng)
            tok = tokenise(inter, vocab, max_len)
            labels = labels + [-1] * (max_len - len(labels))
            real_positions = real_token_positions(labels)
            ids_list.append(torch.tensor(tok["input_ids"], dtype=torch.long))
            mask_list.append(torch.tensor(tok["attention_mask"], dtype=torch.long))
            positions_list.append(real_positions)

        ids = torch.stack(ids_list, dim=0).to(device)
        mask = torch.stack(mask_list, dim=0).to(device)
        _, cache = model.forward(ids, mask, causal=True, return_all=True)
        attn = cache["attn_weights"].detach().cpu().numpy()  # [batch, heads, target_len, source_len]
        n_heads = attn.shape[1]

        for b_idx, real_positions in enumerate(positions_list):
            for step, pos in enumerate(real_positions):
                if pos >= attn.shape[-1]:
                    continue
                valid_len = pos + 1  # causal attention only sees positions <= pos
                for h in range(n_heads):
                    if h not in accum:
                        accum[h] = {"to_cls": [], "to_prev_state": [], "to_prev_bit": [], "to_self": [],
                                    "attention_normalized_entropy": [], "attention_concentration": []}

                    _, norm_ent, conc = _attention_entropy(attn[b_idx, h, pos], valid_len)
                    accum[h]["attention_normalized_entropy"].append(norm_ent)
                    accum[h]["attention_concentration"].append(conc)
                    accum[h]["to_cls"].append(float(attn[b_idx, h, pos, 0]))
                    accum[h]["to_self"].append(float(attn[b_idx, h, pos, pos]))

                    if step > 0:
                        accum[h]["to_prev_state"].append(float(attn[b_idx, h, pos, pos - 1]))
                        accum[h]["to_prev_bit"].append(float(attn[b_idx, h, pos, pos - 2]))
            count += 1

    rows = []
    for h, vals in accum.items():
        row = {"head": h, "n_strings": count}
        for name, scores in vals.items():
            row[name] = float(np.mean(scores)) if scores else float("nan")
        rows.append(row)
    return rows

In [ ]:
def evaluate_component_ablations(model, loader, model_kind, device, s2l=None):
    ablations = {
        "none": {},
        "zero_attention_output": {"ablate_attn": True},
        "zero_mlp_output": {"ablate_mlp": True},
        "zero_positional_encoding": {"ablate_pos": True},
    }
    if model_kind.startswith("cot"):
        ablations["zero_state_token_embeddings"] = {"ablate_state_embeddings": True}

    rows = []
    for name, kwargs in ablations.items():
        if model_kind == "direct":
            acc = eval_direct_accuracy(model, loader, device, **kwargs)
        else:
            acc = eval_cot_final_answer_accuracy(model, loader, device, s2l, **kwargs)
        rows.append({"ablation": name, "metric": "final_answer_accuracy", "value": acc})
    return rows

In [ ]:
def save_rows_csv(path, rows):
    if not rows:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df = pd.DataFrame(rows)
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)


def build_model_for_kind(task_name, model_kind, max_len):
    task = TASKS[task_name]
    if model_kind == "direct":
        vocab = BASE_VOCAB
        n_classes = task["n_classes"]
        state_ids = []
    else:
        vocab = make_cot_vocab(task["n_states"])
        n_classes = task["n_states"]
        state_ids = [vocab[f"s{i}"] for i in range(task["n_states"])]

    model = OneBlockTransformer(
        vocab_size=len(vocab), d_model=D_MODEL, n_heads=N_HEADS, n_classes=n_classes,
        max_len=max(512, max_len + 5), dropout=DROPOUT, pad_id=vocab["[PAD]"], state_token_ids=state_ids,
    )
    return model, vocab


def scratch_mode_of(model_kind):
    return "true" if model_kind == "cot_true" else "pad"


def evaluate_model_on_split(model, model_kind, strings, labels, task_name, max_string_len, seed):
    task = TASKS[task_name]
    transition_fn = task["transition_fn"]
    s2l = task["s2l"]
    n_states = task["n_states"]
    cot_vocab = make_cot_vocab(n_states)

    if model_kind == "direct":
        loader = make_direct_loader(strings, labels, max_base_len(max_string_len), BATCH_SIZE, shuffle=False)
        return {"final_answer_accuracy": eval_direct_accuracy(model, loader, DEVICE)}

    scratch_mode = scratch_mode_of(model_kind)
    loader = make_cot_loader(strings, transition_fn, cot_vocab, max_cot_len(max_string_len), scratch_mode, n_states, BATCH_SIZE, seed, shuffle=False)
    return {
        "state_accuracy": eval_cot_state_accuracy(model, loader, DEVICE),
        "final_answer_accuracy": eval_cot_final_answer_accuracy(model, loader, DEVICE, s2l),
        "transition_consistency": eval_transition_consistency(
            model, strings[:1000], transition_fn, cot_vocab, max_cot_len(max_string_len), DEVICE, scratch_mode, n_states, seed=seed,
        ),
    }


def evaluate_per_length(model, model_kind, task_name, seed):
    task = TASKS[task_name]
    rows = []
    for length in range(PER_LENGTH_MIN, PER_LENGTH_MAX + 1):
        X_len, y_len = split_xy(generate_dataset(
            length, length, PER_LENGTH_SAMPLES, task["label_fn"], task_name, task["n_classes"], seed=seed + 50_000 + length,
        ))
        metrics = evaluate_model_on_split(model, model_kind, X_len, y_len, task_name, length, seed)
        for metric, value in metrics.items():
            rows.append({"task": task_name, "seed": seed, "model": model_kind, "length": length, "metric": metric, "value": value})
    return rows


def train_one_model(task_name, model_kind, X_train, y_train):
    task = TASKS[task_name]
    transition_fn = task["transition_fn"]
    n_states = task["n_states"]

    max_model_len = max_base_len(TRAIN_MAX_LEN) if model_kind == "direct" else max_cot_len(TRAIN_MAX_LEN)
    model, vocab = build_model_for_kind(task_name, model_kind, max_model_len)

    if model_kind == "direct":
        train_loader = make_direct_loader(X_train, y_train, max_base_len(TRAIN_MAX_LEN), BATCH_SIZE, shuffle=True)
        model = train_direct(model, train_loader, N_EPOCHS, LR, DEVICE)
    else:
        scratch_mode = scratch_mode_of(model_kind)
        train_loader = make_cot_loader(X_train, transition_fn, vocab, max_cot_len(TRAIN_MAX_LEN), scratch_mode, n_states, BATCH_SIZE, seed=0, shuffle=True)
        model = train_cot_state(model, train_loader, N_EPOCHS, LR, DEVICE, n_states)

    return model, vocab


def run_single_task_seed(task_name, seed):
    print("=" * 80)
    print(f"Running task={task_name}, seed={seed}")
    print("=" * 80)
    set_seed(seed)

    task = TASKS[task_name]
    X_train, y_train = split_xy(generate_dataset(TRAIN_MIN_LEN, TRAIN_MAX_LEN, TRAIN_SAMPLES, task["label_fn"], task_name, task["n_classes"], seed))
    X_iid, y_iid = split_xy(generate_dataset(TRAIN_MIN_LEN, TRAIN_MAX_LEN, IID_TEST_SAMPLES, task["label_fn"], task_name, task["n_classes"], seed + 1))
    X_ood, y_ood = split_xy(generate_dataset(OOD_MIN_LEN, OOD_MAX_LEN, OOD_TEST_SAMPLES, task["label_fn"], task_name, task["n_classes"], seed + 2))

    result_rows = []
    per_length_rows = []
    attention_rows = []
    ablation_rows = []
    geometry_rows = []

    for model_kind in MODEL_KINDS:
        print(f"--- Training model: {model_kind} ---")
        model, vocab = train_one_model(task_name, model_kind, X_train, y_train)

        # Main IID / OOD-length evaluation
        eval_sets = {
            "iid": (X_iid, y_iid, TRAIN_MAX_LEN),
            "ood_length": (X_ood, y_ood, OOD_MAX_LEN),
        }
        for split_name, (X_eval, y_eval, max_len_eval) in eval_sets.items():
            metrics = evaluate_model_on_split(model, model_kind, X_eval, y_eval, task_name, max_len_eval, seed)
            for metric, value in metrics.items():
                result_rows.append({"task": task_name, "seed": seed, "model": model_kind, "split": split_name, "metric": metric, "value": value})

        # Per-length OOD curve
        print(f"Evaluating per-length curve for {model_kind}...")
        per_length_rows.extend(evaluate_per_length(model, model_kind, task_name, seed))

        # Representational geometry analysis
        geom_strings = X_iid[:MAX_GEOMETRY_STRINGS]
        geom_labels = y_iid[:MAX_GEOMETRY_STRINGS]
        if model_kind == "direct":
            geom_mode, geom_vocab, geom_max_len, geom_scratch = "direct", BASE_VOCAB, max_base_len(TRAIN_MAX_LEN), "none"
        else:
            geom_mode, geom_vocab, geom_max_len, geom_scratch = "cot", vocab, max_cot_len(TRAIN_MAX_LEN), scratch_mode_of(model_kind)

        print(f"Running geometry analysis for {model_kind}...")
        geometry_rows.extend(run_representational_geometry_analysis(
            model, geom_strings, geom_labels, task["transition_fn"], geom_vocab, geom_max_len,
            geom_mode, DEVICE, task_name, model_kind, seed, geom_scratch, task["n_states"],
        ))

        # Attention-weight analysis)
        if model_kind.startswith("cot"):
            scratch_mode = scratch_mode_of(model_kind)
            att_rows = attention_routing_analysis(
                model, X_iid[:MAX_ATTENTION_STRINGS], task["transition_fn"], vocab,
                max_cot_len(TRAIN_MAX_LEN), DEVICE, scratch_mode, task["n_states"], seed,
            )
            for row in att_rows:
                row.update({"task": task_name, "seed": seed, "model": model_kind})
            attention_rows.extend(att_rows)

        # Component ablations on the IID split
        if model_kind == "direct":
            loader = make_direct_loader(X_iid, y_iid, max_base_len(TRAIN_MAX_LEN), BATCH_SIZE, shuffle=False)
            abl_rows = evaluate_component_ablations(model, loader, model_kind, DEVICE)
        else:
            scratch_mode = scratch_mode_of(model_kind)
            loader = make_cot_loader(X_iid, task["transition_fn"], vocab, max_cot_len(TRAIN_MAX_LEN), scratch_mode, task["n_states"], BATCH_SIZE, seed, shuffle=False)
            abl_rows = evaluate_component_ablations(model, loader, model_kind, DEVICE, s2l=task["s2l"])
        for row in abl_rows:
            row.update({"task": task_name, "seed": seed, "model": model_kind, "split": "iid"})
        ablation_rows.extend(abl_rows)

    save_rows_csv(RESULTS_CSV, result_rows)
    save_rows_csv(PER_LENGTH_CSV, per_length_rows)
    save_rows_csv(ATTENTION_CSV, attention_rows)
    save_rows_csv(ABLATION_CSV, ablation_rows)
    save_rows_csv(GEOMETRY_CSV, geometry_rows)

    print(f"Finished task={task_name}, seed={seed}")


def run_all_experiments():
    for task_name in TASK_NAMES:
        for seed in SEEDS:
            run_single_task_seed(task_name, seed)

# run_all_experiments()